# Apex Retail Intelligence

## Gold Layer

### Objective
Create Dimension and Fact tables for business reporting.

## Project Pipeline

```text
                Apex Retail Intelligence Pipeline

 Historical CSV          Incremental CSV
        │                      │
        └──────────┬───────────┘
                   ▼
            Raw Landing Layer
                   │
                   ▼
             Bronze Layer
            (Delta Storage)
                   │
                   ▼
              Silver Layer
      (Cleaning + MERGE + SCD)
                   │
                   ▼
               Gold Layer
      (Dimensions & Fact Tables)
                   │
                   ▼
              KPI Dashboard
```

In [0]:
# ============================================================
# Step 1 : Create Gold Layer Paths
# ============================================================

GOLD_PATH = "/Workspace/Users/akashpatra788@gmail.com/Apex_Retail_Intelligence/04_Gold"

DIM_CUSTOMER_PATH = f"{GOLD_PATH}/dim_customer"
DIM_PRODUCT_PATH = f"{GOLD_PATH}/dim_product"
DIM_DATE_PATH = f"{GOLD_PATH}/dim_date"
FACT_SALES_PATH = f"{GOLD_PATH}/fact_sales"

print("Gold paths created successfully.")

Gold paths created successfully.


In [0]:
# ============================================================
# Step 1.1 : Define Silver Path
# ============================================================

SILVER_PATH = "/Workspace/Users/akashpatra788@gmail.com/Apex_Retail_Intelligence/03_Silver"

print(SILVER_PATH)

/Workspace/Users/akashpatra788@gmail.com/Apex_Retail_Intelligence/03_Silver


In [0]:
# ============================================================
# Step 2 : Load Silver Delta Tables
# ============================================================

customer_gold_df = spark.read.format("delta").load(
    "dbfs:/Workspace/Users/akashpatra788@gmail.com/Apex_Retail_Intelligence/03_Silver/customer"
)

product_gold_df = spark.read.format("delta").load(
    "dbfs:/Workspace/Users/akashpatra788@gmail.com/Apex_Retail_Intelligence/03_Silver/product"
)

sales_gold_df = spark.read.format("delta").load(
    "dbfs:/Workspace/Users/akashpatra788@gmail.com/Apex_Retail_Intelligence/03_Silver/sales"
)

print("Silver tables loaded successfully.")

Silver tables loaded successfully.


## Step 3 : Preview Silver Tables

In this step, I am displaying the Silver Layer datasets.

This helps verify that the cleaned data has been loaded successfully before creating the Gold Layer dimensions and fact table.

In [0]:
# ============================================================
# Step 3 : Preview Silver Tables
# ============================================================

display(customer_gold_df.limit(5))
display(product_gold_df.limit(5))
display(sales_gold_df.limit(5))

customer_id,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,occupation,customer_zip_code,customer_city,customer_state,ingested_at,customer_sk
1,56,Other,High,No,0,No,Divorced,3,Bachelor's,Self-Employed,37848,City D,State Y,2026-08-08T17:01:44.915Z,0
2,69,Female,Medium,No,2,No,Married,2,PhD,Unemployed,44896,null,State X,2026-08-08T17:01:44.915Z,1
3,46,Female,Low,No,5,No,Married,3,Bachelor's,Self-Employed,11816,City B,State X,2026-08-08T17:01:44.915Z,2
4,32,Female,Low,No,0,No,Divorced,2,Master's,Employed,78604,City A,State Y,2026-08-08T17:01:44.915Z,3
6,25,Other,Medium,Yes,4,Yes,Divorced,0,Bachelor's,Unemployed,54549,City D,State Z,2026-08-08T17:01:44.915Z,4


product_id,product_name,product_brand,product_category,product_rating,product_review_count,product_stock,product_return_rate,product_size,product_weight,product_color,product_material,product_manufacture_date,product_expiry_date,product_shelf_life,unit_price,ingested_at,product_sk
100,Product D,Brand Y,Toys,2.4,521,87,0.35,Large,6.68,Red,Wood,2018-09-23 05:24:09,2023-02-17 15:50:47,210,629.45,2026-08-08T17:01:53.745Z,0
10000,Product E,Brand Y,Furniture,2.4,644,34,0.4,Large,3.54,Red,Wood,2021-07-21 02:16:30,2025-03-27 16:55:32,1345,914.04,2026-08-08T17:01:53.745Z,1
10001,Product B,Brand X,Groceries,2.1,163,66,0.43,Medium,4.94,White,Glass,2021-03-26 02:38:08,2024-01-10 15:34:29,1020,446.16,2026-08-08T17:01:53.745Z,2
10002,Product C,Brand Y,Toys,1.2,316,76,0.36,Small,6.81,Red,Wood,2020-06-21 00:37:29,2020-10-20 19:38:50,121,523.74,2026-08-08T17:01:53.745Z,3
10003,Product D,Brand Y,Clothing,4.9,701,99,0.31,Small,3.33,White,Metal,2022-04-16 13:38:13,2027-06-29 16:06:17,1900,467.44,2026-08-08T17:01:53.745Z,4


transaction_id,transaction_date,customer_id,product_id,quantity,unit_price,discount_applied,payment_method,store_location,transaction_hour,day_of_week,week_of_year,month_of_year,total_sales,promotion_id,promotion_type,holiday_season,season,weekend,ingested_at,sales_sk
191823,2021-11-08 21:20:37,23,8715,9,512.97,0.02,Debit Card,null,11,Sunday,11,11,8820.87,462,20% Off,No,Fall,No,2026-08-08T17:01:47.668Z,0
974720,2020-02-08 02:45:45,26,414,2,687.35,0.04,Mobile Payment,Location B,3,Wednesday,6,9,1639.63,786,20% Off,No,Spring,No,2026-08-08T17:01:47.668Z,1
577038,2021-03-27 13:44:19,29,4642,5,833.11,0.32,Credit Card,Location C,10,Monday,46,4,1459.27,334,20% Off,Yes,Winter,No,2026-08-08T17:01:47.668Z,2
765484,2020-07-16 08:00:45,48,8087,8,719.79,0.13,Cash,Location B,20,Friday,47,7,8729.63,877,Flash Sale,No,Spring,No,2026-08-08T17:01:47.668Z,3
795563,2020-02-21 21:15:33,92,8534,8,607.74,0.5,Debit Card,Location B,5,Saturday,17,11,1340.42,543,Buy One Get One Free,No,Spring,Yes,2026-08-08T17:01:47.668Z,4


In [0]:
# ============================================================
# Step 4 : Check Dataset Size
# ============================================================

print("Customer :", customer_gold_df.count())
print("Product  :", product_gold_df.count())
print("Sales    :", sales_gold_df.count())

Customer : 1050
Product  : 1043
Sales    : 2000


## Step 5 : Create Customer Dimension

In this step, I am creating the Customer Dimension table from the Silver Layer.

Only the required customer attributes are selected for analytical reporting.

In [0]:
# ============================================================
# Step 5 : Create Customer Dimension
# ============================================================

dim_customer_df = customer_gold_df.select(
    "customer_sk",
    "customer_id",
    "age",
    "gender",
    "income_bracket",
    "customer_city",
    "customer_state"
)

display(dim_customer_df.limit(5))

customer_sk,customer_id,age,gender,income_bracket,customer_city,customer_state
0,1,56,Other,High,City D,State Y
1,2,69,Female,Medium,null,State X
2,3,46,Female,Low,City B,State X
3,4,32,Female,Low,City A,State Y
4,6,25,Other,Medium,City D,State Z


## Step 6 : Create Product Dimension

In this step, I am creating the Product Dimension table from the Silver Layer.

Only the required product attributes are selected for reporting and analysis.

In [0]:
# ============================================================
# Step 6 : Create Product Dimension
# ============================================================

dim_product_df = product_gold_df.select(
    "product_sk",
    "product_id",
    "product_name",
    "product_brand",
    "product_category",
    "unit_price"
)

display(dim_product_df.limit(5))

product_sk,product_id,product_name,product_brand,product_category,unit_price
0,100,Product D,Brand Y,Toys,629.45
1,10000,Product E,Brand Y,Furniture,914.04
2,10001,Product B,Brand X,Groceries,446.16
3,10002,Product C,Brand Y,Toys,523.74
4,10003,Product D,Brand Y,Clothing,467.44


## Step 7 : Create Date Dimension

In this step, I am creating the Date Dimension from the Sales dataset.

This dimension stores useful calendar attributes for reporting.

In [0]:
# ============================================================
# Step 7 : Create Date Dimension
# ============================================================

from pyspark.sql.functions import (
    to_date,
    year,
    month,
    weekofyear,
    dayofmonth,
    dayofweek
)

dim_date_df = (
    sales_gold_df
    .select("transaction_date")
    .distinct()
    .withColumn("date", to_date("transaction_date"))
    .withColumn("year", year("date"))
    .withColumn("month", month("date"))
    .withColumn("week", weekofyear("date"))
    .withColumn("day", dayofmonth("date"))
    .withColumn("day_of_week", dayofweek("date"))
)

display(dim_date_df.limit(5))

transaction_date,date,year,month,week,day,day_of_week
2021-03-09 04:28:26,2021-03-09,2021,3,10,9,3
2021-03-10 18:01:53,2021-03-10,2021,3,10,10,4
2022-03-18 20:09:55,2022-03-18,2022,3,11,18,6
2023-09-13 21:48:59,2023-09-13,2023,9,37,13,4
2023-12-11 19:43:03,2023-12-11,2023,12,50,11,2


## Step 8 : Create Fact Sales Table

In this step, I am creating the Fact Sales table.

The fact table combines sales transactions with customer and product surrogate keys for analytical reporting.

In [0]:
# ============================================================
# Step 8 : Create Fact Sales Table
# ============================================================

fact_sales_df = sales_gold_df.select(
    "sales_sk",
    "transaction_id",
    "customer_id",
    "product_id",
    "transaction_date",
    "quantity",
    "unit_price",
    "discount_applied",
    "total_sales"
)

display(fact_sales_df.limit(5))

sales_sk,transaction_id,customer_id,product_id,transaction_date,quantity,unit_price,discount_applied,total_sales
0,191823,23,8715,2021-11-08 21:20:37,9,512.97,0.02,8820.87
1,974720,26,414,2020-02-08 02:45:45,2,687.35,0.04,1639.63
2,577038,29,4642,2021-03-27 13:44:19,5,833.11,0.32,1459.27
3,765484,48,8087,2020-07-16 08:00:45,8,719.79,0.13,8729.63
4,795563,92,8534,2020-02-21 21:15:33,8,607.74,0.5,1340.42


## Step 9 : Save Gold Tables

In this step, I am storing all Gold Layer tables in Delta format.

In [0]:
# ============================================================
# Step 9 : Save Gold Tables
# ============================================================

dim_customer_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("dbfs:/Workspace/Users/akashpatra788@gmail.com/Apex_Retail_Intelligence/04_Gold/dim_customer")

dim_product_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("dbfs:/Workspace/Users/akashpatra788@gmail.com/Apex_Retail_Intelligence/04_Gold/dim_product")

dim_date_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("dbfs:/Workspace/Users/akashpatra788@gmail.com/Apex_Retail_Intelligence/04_Gold/dim_date")

fact_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("dbfs:/Workspace/Users/akashpatra788@gmail.com/Apex_Retail_Intelligence/04_Gold/fact_sales")

print("Gold tables saved successfully.")

Gold tables saved successfully.


## Step 10 : Validate Gold Layer

In this step, I am validating the Gold Layer by checking the record count of each Gold table.

In [0]:
# ============================================================
# Step 10 : Gold Validation
# ============================================================

validation = [

    ("Customer Dimension", dim_customer_df.count()),

    ("Product Dimension", dim_product_df.count()),

    ("Date Dimension", dim_date_df.count()),

    ("Fact Sales", fact_sales_df.count())

]

validation_df = spark.createDataFrame(
    validation,
    ["Table", "Records"]
)

display(validation_df)

Table,Records
Customer Dimension,1050
Product Dimension,1043
Date Dimension,1937
Fact Sales,2000


## Step 11 : Gold Layer Summary

The Gold Layer has been completed successfully.

The following tables were created:

- Customer Dimension
- Product Dimension
- Date Dimension
- Fact Sales

The Gold Layer is now ready for KPI analysis.

In [0]:
# ============================================================
# Step 11 : Gold Layer Summary
# ============================================================

print("Gold Layer completed successfully.")

print("Customer Dimension :", dim_customer_df.count())
print("Product Dimension  :", dim_product_df.count())
print("Date Dimension     :", dim_date_df.count())
print("Fact Sales         :", fact_sales_df.count())

Gold Layer completed successfully.
Customer Dimension : 1050
Product Dimension  : 1043
Date Dimension     : 1937
Fact Sales         : 2000


## Notebook Summary

✔ Customer Dimension Created

✔ Product Dimension Created

✔ Date Dimension Created

✔ Fact Sales Created

✔ Ready for KPI Analysis